# GAN and VAE generation on MNIST

This notebook runs the same experiment as `python -m gan_vae`. A 20-dimensional MLP VAE and GAN share a frozen CNN evaluator. The saved benchmark uses the full preset; the default here is a shorter MNIST run.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd() if (Path.cwd() / "gan_vae").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from gan_vae.experiment import preset_config, run_experiment
from gan_vae.models import VAE, Generator, Discriminator, MNISTJudge
from IPython.display import Image, display


## Architecture and objective

The VAE uses an encoder with 400 hidden units, Gaussian reparameterization, and a Tanh decoder. Its loss is pixel squared error plus KL divergence, both summed per image. The GAN uses the original MLP widths and non-saturating generator objective, with BCE computed directly from discriminator logits. Equal latent dimensions and epochs leave parameter counts and training costs different.


In [ ]:
models = {"vae": VAE(), "generator": Generator(), "discriminator": Discriminator(), "judge": MNISTJudge()}
for name, model in models.items():
    print(name, sum(parameter.numel() for parameter in model.parameters()), "parameters")


## Run the experiment

`quick` trains on 10,000 official training images for 3 classifier epochs and 5 epochs per generative model, then evaluates 1,000 generated images per model. `full` uses all 60,000 training images, 5 classifier epochs, 50 epochs per generative model and 5,000 evaluation images per model. Both use the official test partition only after training.

Set `GAN_VAE_NOTEBOOK_PRESET=full` before starting the notebook to run the longer experiment. The optional `smoke` setting uses random tensors for an offline execution check and produces no MNIST performance evidence. Set `GAN_VAE_NOTEBOOK_DOWNLOAD=1` to permit a first MNIST download, or run the CLI once with `--download`.


In [ ]:
preset = os.environ.get("GAN_VAE_NOTEBOOK_PRESET", "quick")
config = preset_config(preset)
output = ROOT / "results" / "notebook"
metrics = run_experiment(config, output=output, data_root=ROOT / "data",
                         download=os.environ.get("GAN_VAE_NOTEBOOK_DOWNLOAD") == "1",
                         overwrite=True)


## Classifier diagnostics

Classifier test accuracy describes the evaluator on real held-out digits. Generated-image confidence describes its predictions, rather than the probability that an image is realistic. Predicted-class entropy measures balance across ten classifier labels and cannot establish diversity within each class.


In [ ]:
print("Run:", metrics["run_type"])
print("Classifier test accuracy:", metrics["judge_test_accuracy"])
for name, row in metrics["evaluation"].items():
    print(name, "confidence", round(row["mean_max_probability"], 4),
          "entropy", round(row["predicted_class_entropy_nats"], 4),
          "CNN distance", round(row.get("cnn_frechet_distance", 0), 4))


## Generated images and latent interpolation

The two models receive the same seeded latent vectors. The interpolation uses the same two endpoints and ten evenly spaced positions for both models. Inspect the actual samples alongside the diagnostics.


In [ ]:
for name in ["vae_samples.png", "gan_samples.png", "interpolation.png"]:
    display(Image(filename=str(output / name)))


## Training and predicted classes

VAE and GAN losses have different definitions and scales. Their values should be read within each model, without treating lower loss across the two panels as a model ranking. Class counts come from the same images used in the distance calculation.


In [ ]:
for name in ["training.png", "predicted_classes.png"]:
    display(Image(filename=str(output / name)))


## Reading the comparison

The distance uses 128 features from this project's frozen MNIST CNN. It is not standard Inception FID and cannot be compared with published FID numbers. A disjoint real-versus-real comparison gives context for finite-sample distance; uniform random pixels provide a negative control. Neither control establishes an absolute quality threshold.

The experiment uses fixed budgets and final checkpoints, with no hyperparameter or checkpoint selection on the test set. One seed does not establish consistent superiority. Cross-device or cross-version runs may differ despite deterministic settings.


In [ ]:
for limitation in metrics["limitations"]:
    print("-", limitation)
print("Saved checkpoint:", output / "models.pt")
print("Settings, source hashes and results:", output / "metrics.json")
